In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal as ss

In [ ]:
PARQ_PATH = Path('../sample_dataset/processed_data/cleaned_parquets/20220730-0001_c.parquet')

In [ ]:
df = pd.read_parquet(PARQ_PATH)
df.columns = df.columns.astype('int16')

In [ ]:
df_norm = (df.T - df.T.min())/(df.T.max() - df.T.min())
df_norm = df_norm.T

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

sig_range = [0, 100]

ax.plot(
    df_norm.iloc[sig_range[0]:sig_range[1]].T
)

fig.show()

In [ ]:
test_sig = df_norm.iloc[0]

test_sig.name

In [ ]:
# Plot a signal and highlight peak onset, peak max, and tail start
fig, ax = plt.subplots(figsize=(15,8))

sig_index = 10

# Plot the normalized signal
ax.plot(
    df_norm.iloc[sig_index].T
)

# Find peak data
peak_indices, peak_props = ss.find_peaks(df_norm.iloc[sig_index], height=0.1, prominence=0.05)

# Plot all peaks
for p in peak_indices:
    ax.scatter(
        x=p,
        y=df_norm.iloc[sig_index][p] + 0.02,
        marker='v',
        c='red',
        label="peak"
    )
    
    tail_start = p + 5
    ax.scatter(
        x=tail_start,
        y=df_norm.iloc[sig_index][tail_start] + 0.02,
        marker='v',
        c='blue',
        label="tail start"
    )

# Plot peak onset
l_bases = peak_props['left_bases']
for l_b in l_bases:
    ax.scatter(
        x=l_b,
        y=df_norm.iloc[sig_index][l_b] - 0.02,
        marker='^',
        c='green',
        label="peak onset"
    )

# Add text with signal ID
ax.text(
    x=0,
    y=1,
    s=f"signal: {df_norm.iloc[sig_index].name}",
    fontfamily='monospace',
    fontsize=14
)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time axis (a.u.)", fontsize=18)
ax.set_ylabel("normalized amplitude (a.u.)", fontsize=18)
ax.legend()

fig.show()

In [ ]:
# TODO: break this function apart
def create_psd_df_for_buffer(
    df: pd.DataFrame, baseline_offset: float = 0.05, peak_prominence: float = 0.05, tail_offset: int = 5
) -> pd.DataFrame:

    df = df + baseline_offset

    # normalize input dataframe (faster to normalize each row in loop below?)
    df_norm = (df.T - df.T.min()) / (df.T.max() - df.T.min())
    df_norm = df_norm.T

    # create empty psd dict (better to create DataFrame and append/concat?)
    psd_dict = {}

    # For each row in df:
    for i in range(df.shape[0]):
        signal_norm = df_norm.iloc[i]
        signal_name = signal_norm.name
        signal_raw = df.loc[signal_name]

        # Get peak data from normalized signal
        peaks, props = ss.find_peaks(
            signal_norm, prominence=peak_prominence, height=0.1
        )

        # Define integral indices
        peak_onset = props["left_bases"][0]
        tail_start = peaks[0] + tail_offset

        # Calculate integrals (sums) for raw signal
        peak_integral = signal_raw[peak_onset:tail_start].sum()
        tail_integral = signal_raw[tail_start:].sum()
        total_integral = peak_integral + tail_integral

        # Add relevant data to dict
        psd_dict[signal_name] = np.array(
            [
                tail_integral,
                total_integral,
                signal_raw.max(),  # signal amplitude in volts
            ]
        )
    
    # Create a DataFrame from the dict
    psd_df = pd.DataFrame(psd_dict)
    psd_df = psd_df.T
    psd_df.columns = ["tail", "total", "amplitude (V)"]
    psd_df["psd"] = psd_df["tail"] / psd_df["total"]

    return psd_df


In [ ]:
start = time.time()
psd_df = create_psd_df_for_buffer(
    df,
    baseline_offset=0.01
)

print(f"Calculated PSD values for {df.shape[0]} signals. [{time.time() - start:.2f} s]")

In [ ]:
# Plot PSD (tail vs. total; psd vs. amplitude; psd histogram)
fig, axs = plt.subplots(1, 3, figsize=(18,6))

label_fs = 14

axs[0].scatter(
    psd_df['total'],
    psd_df['tail'],
    marker='.',
    s=1
)

axs[0].set_xlabel("total integral (a.u.)", fontsize=label_fs)
axs[0].set_ylabel("tail integral (a.u.)", fontsize=label_fs)

axs[1].scatter(
    psd_df['amplitude (V)'],
    psd_df['psd'],
    marker='.',
    s=1
)

axs[1].set_xlabel("pulse amplitude (V)", fontsize=label_fs)
axs[1].set_ylabel("tail / total (a.u.)", fontsize=label_fs)


n_bins = 100
axs[2].hist(
    psd_df['psd'],
    bins=n_bins,
    linewidth=2,
    histtype='step',
    orientation='horizontal'
)

axs[2].text(
    x=0.75,
    y=0.95,
    s=f"n_bins = {n_bins}",
    transform=axs[2].transAxes
)

axs[2].set_xlabel("counts", fontsize=label_fs)
axs[2].set_ylabel("tail / total (a.u.)", fontsize=label_fs)

fig.set_facecolor('white')

fig.show()